# Job Lifecycle Fact Table Loader

This notebook maintains the `warehouse.fact_job_lifecycle` fact table using standardized batch processing.

**Purpose**: Track job state transitions and lifecycle events

**Key Features**:
* **Fact Grain**: One row per job state change event
* **Lifecycle Transitions**: JOB_CREATED, JOB_ACTIVATED, JOB_DEACTIVATED, JOB_SOFT_DELETED, JOB_RESTORED, JOB_UPDATED, JOB_DELETED
* **Duration Metrics**: Time spent in previous state before transition
* **SCD2-Aware Joins**: Point-in-time dimension joins using effective date windows
* **Incremental Processing**: Batch-based loading with metadata tracking
* **Robust Error Handling**: Comprehensive validation and error logging

**Architecture**:
- **Source**: Silver layer change tracking (`workspace.silver.silver_job_changes`)
- **Dimensions**: SCD2 job dimension, company, location, sector, role, source dimensions
- **Target**: `workspace.warehouse.fact_job_lifecycle`
- **Metadata**: `workspace.metadata.fact_job_lifecycle_refresh_log`

**Processing Logic**:
1. Extract change events incrementally by batch_id
2. Classify lifecycle transitions (created, activated, deactivated, etc.)
3. Calculate duration metrics (time in previous state)
4. Resolve foreign keys with SCD2 time-travel joins
5. Generate surrogate keys and insert new fact records
6. Log metadata and validate results

In [0]:
dbutils.widgets.text("force_full_refresh", "false", "Force Full Refresh (true/false)")

force_full_refresh = dbutils.widgets.get("force_full_refresh").strip().lower() == "true"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, TimestampType, BooleanType
from pyspark.sql.window import Window
from datetime import datetime
import json

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Source and target tables
SOURCE_CHANGES_TABLE = f"{SILVER_SCHEMA}.silver_job_changes"
JOB_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_job"
COMPANY_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_company"
LOCATION_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_location"
SECTOR_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_sector"
ROLE_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_role"
SOURCE_DIM_TABLE = f"{WAREHOUSE_SCHEMA}.dim_source"
TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.fact_job_lifecycle"
METADATA_TABLE = f"{METADATA_SCHEMA}.fact_job_lifecycle_refresh_log"

# Generate run ID
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {run_id}")
print(f"Source table: {SOURCE_CHANGES_TABLE}")
print(f"Force full refresh: {force_full_refresh}")

In [0]:
# Define fact table schema
fact_schema = StructType([
    StructField("fact_job_lifecycle_sk", LongType(), False, metadata={"comment": "Surrogate key"}),
    StructField("change_id", StringType(), False, metadata={"comment": "Unique change event ID from source"}),
    StructField("job_sk", LongType(), False, metadata={"comment": "FK to dim_job (SCD2 time-travel)"}),
    StructField("company_sk", LongType(), False, metadata={"comment": "FK to dim_company"}),
    StructField("location_sk", LongType(), False, metadata={"comment": "FK to dim_location"}),
    StructField("sector_sk", LongType(), False, metadata={"comment": "FK to dim_sector"}),
    StructField("role_sk", LongType(), False, metadata={"comment": "FK to dim_role"}),
    StructField("source_sk", LongType(), False, metadata={"comment": "FK to dim_source"}),
    StructField("change_date_sk", IntegerType(), False, metadata={"comment": "Date key (YYYYMMDD format)"}),
    StructField("change_timestamp", TimestampType(), False, metadata={"comment": "When the change occurred"}),
    StructField("change_type", StringType(), False, metadata={"comment": "INSERT, UPDATE, DELETE, RESTORE"}),
    StructField("lifecycle_event_type", StringType(), False, metadata={"comment": "Classified event type"}),
    StructField("days_in_previous_state", IntegerType(), True, metadata={"comment": "Days in state before this change"}),
    StructField("batch_id", StringType(), False, metadata={"comment": "Source batch that detected change"})
])

# Define metadata table schema
metadata_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("changes_extracted", IntegerType(), False),
    StructField("new_facts_inserted", IntegerType(), False),
    StructField("facts_skipped", IntegerType(), False),
    StructField("force_full_refresh", BooleanType(), False),
    StructField("run_timestamp", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True)
])

# Create target table if not exists
if not spark.catalog.tableExists(TARGET_TABLE):
    spark.createDataFrame([], schema=fact_schema).write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"Created fact table: {TARGET_TABLE}")
else:
    print(f"Fact table exists: {TARGET_TABLE}")

# Create metadata table if not exists
if not spark.catalog.tableExists(METADATA_TABLE):
    spark.createDataFrame([], schema=metadata_schema).write.format("delta").saveAsTable(METADATA_TABLE)
    print(f"Created metadata table: {METADATA_TABLE}")
else:
    print(f"Metadata table exists: {METADATA_TABLE}")

In [0]:
print("Extracting lifecycle change events...", end=" ")

# Get last processed batch_id for incremental loading
if not force_full_refresh and spark.catalog.tableExists(METADATA_TABLE):
    try:
        last_batch_result = spark.sql(f"""
            SELECT MAX(batch_id) as last_batch
            FROM {TARGET_TABLE}
            WHERE batch_id IS NOT NULL
        """).collect()
        last_batch_id = last_batch_result[0]['last_batch'] if last_batch_result[0]['last_batch'] else None
    except Exception:
        # Table exists but doesn't have batch_id column (old schema) - do full refresh
        last_batch_id = None
else:
    last_batch_id = None

print(f"(incremental from batch: {last_batch_id})" if last_batch_id else "(full refresh)", end=" ")

# Load change events
changes_df = spark.table(SOURCE_CHANGES_TABLE).filter(
    (F.col("enterprise_job_id").isNotNull()) &
    (F.col("change_type").isNotNull())
)

# Apply incremental filter if not full refresh
if last_batch_id:
    changes_df = changes_df.filter(F.col("batch_id") > last_batch_id)

changes_count = changes_df.count()
print(f"✓ Extracted {changes_count} change events")

if changes_count == 0:
    print("No new changes to process. Exiting.")
    dbutils.notebook.exit(json.dumps({"status": "success", "message": "No new changes"}))

print("Parsing JSON and classifying transitions...", end=" ")

# Parse old/new values from JSON and classify lifecycle transitions
changes_classified = changes_df.withColumn(
    "old_is_active",
    F.get_json_object(F.col("old_value_json"), "$.is_active")
).withColumn(
    "new_is_active",
    F.get_json_object(F.col("new_value_json"), "$.is_active")
).withColumn(
    "old_soft_delete",
    F.get_json_object(F.col("old_value_json"), "$.soft_delete_flag")
).withColumn(
    "new_soft_delete",
    F.get_json_object(F.col("new_value_json"), "$.soft_delete_flag")
).withColumn(
    "lifecycle_event_type",
    F.when(F.col("change_type") == "INSERT", F.lit("JOB_CREATED"))
    .when(F.col("change_type") == "DELETE", F.lit("JOB_DELETED"))
    .when(
        (F.col("old_is_active") == "false") & (F.col("new_is_active") == "true"),
        F.lit("JOB_ACTIVATED")
    )
    .when(
        (F.col("old_is_active") == "true") & (F.col("new_is_active") == "false"),
        F.lit("JOB_DEACTIVATED")
    )
    .when(
        (F.col("old_soft_delete") == "false") & (F.col("new_soft_delete") == "true"),
        F.lit("JOB_SOFT_DELETED")
    )
    .when(
        (F.col("old_soft_delete") == "true") & (F.col("new_soft_delete") == "false"),
        F.lit("JOB_RESTORED")
    )
    .when(F.col("change_type") == "UPDATE", F.lit("JOB_UPDATED"))
    .otherwise(F.lit("UNKNOWN_TRANSITION"))
)

print("✓ Transitions classified")

print("Calculating duration metrics...", end=" ")

# Calculate time in previous state using window functions
window_spec = Window.partitionBy("enterprise_job_id").orderBy("change_timestamp")

changes_with_duration = changes_classified.withColumn(
    "previous_change_timestamp",
    F.lag("change_timestamp").over(window_spec)
).withColumn(
    "days_in_previous_state",
    F.datediff(F.col("change_timestamp"), F.col("previous_change_timestamp"))
)

print("✓ Duration metrics calculated")

In [0]:
print("Resolving foreign keys with SCD2 time-travel...", end=" ")

# Check if prerequisite variable exists (previous cell may have exited early)
try:
    changes_with_duration
except NameError:
    print("✗ No changes to process (upstream cell exited early)")
    dbutils.notebook.exit(json.dumps({"status": "success", "message": "No changes to process"}))

# Load dimension tables
job_dim_df = spark.table(JOB_DIM_TABLE)
role_dim_df = spark.table(ROLE_DIM_TABLE)
source_dim_df = spark.table(SOURCE_DIM_TABLE)

# SCD2 time-travel join to job dimension
# Join on enterprise_job_id and change_timestamp falls within effective window
changes_with_job = changes_with_duration.alias("c").join(
    job_dim_df.alias("j"),
    (F.col("c.enterprise_job_id") == F.col("j.enterprise_job_id")) &
    (F.col("c.change_timestamp") >= F.col("j.effective_from")) &
    ((F.col("c.change_timestamp") < F.col("j.effective_to")) | F.col("j.effective_to").isNull()),
    "left"
).select(
    F.col("c.*"),
    F.coalesce(F.col("j.job_sk"), F.lit(-1)).alias("job_sk"),
    F.coalesce(F.col("j.company_sk"), F.lit(-1)).alias("company_sk"),
    F.coalesce(F.col("j.location_sk"), F.lit(-1)).alias("location_sk"),
    F.coalesce(F.col("j.sector_sk"), F.lit(-1)).alias("sector_sk"),
    F.col("j.canonical_role_id")
)

# Join to role dimension
changes_with_role = changes_with_job.alias("c").join(
    role_dim_df.alias("r"),
    F.col("c.canonical_role_id") == F.col("r.canonical_role_id"),
    "left"
).select(
    F.col("c.*"),
    F.coalesce(F.col("r.role_sk"), F.lit(-1)).alias("role_sk")
).drop("canonical_role_id")

# Join to source dimension
facts_with_fks = changes_with_role.alias("c").join(
    source_dim_df.alias("s"),
    F.col("c.source_name") == F.col("s.source_name"),
    "left"
).select(
    F.col("c.*"),
    F.coalesce(F.col("s.source_sk"), F.lit(-1)).alias("source_sk")
)

# Add change date surrogate key (YYYYMMDD format)
facts_with_fks = facts_with_fks.withColumn(
    "change_date_sk",
    F.date_format(F.col("change_timestamp"), "yyyyMMdd").cast(IntegerType())
)

# Filter out records with missing job FK (can't link to dimension)
facts_valid = facts_with_fks.filter(
    (F.col("job_sk") != -1) & F.col("job_sk").isNotNull()
)

skipped_count = changes_count - facts_valid.count()

print(f"✓ Foreign keys resolved ({skipped_count} records skipped due to missing job FK)")

In [0]:
print(f"Inserting new fact records into {TARGET_TABLE}...", end=" ")

# Check if prerequisite variables exist (previous cells may have exited early)
try:
    facts_valid
    skipped_count
except NameError:
    print("✗ No changes to process (upstream cells exited early)")
    dbutils.notebook.exit(json.dumps({"status": "success", "message": "No changes to process"}))

try:
    # Check if table has the new schema (with change_id column)
    target_columns = [col.name for col in spark.table(TARGET_TABLE).schema.fields]
    has_new_schema = "change_id" in target_columns
    
    if has_new_schema:
        # Check for existing change_ids to avoid duplicates
        existing_change_ids = spark.table(TARGET_TABLE).select("change_id").distinct()
        
        # Anti-join to get only new records
        new_facts = facts_valid.join(
            existing_change_ids,
            facts_valid.change_id == existing_change_ids.change_id,
            "left_anti"
        )
    else:
        # Old schema detected - this is the first load with new schema, load all records
        print("(old schema detected - will replace table)...", end=" ")
        new_facts = facts_valid
    
    new_facts_count = new_facts.count()
    
    if new_facts_count == 0:
        print("✓ No new facts to insert (all change_ids already exist)")
    else:
        # Get current max surrogate key
        max_sk_result = spark.sql(f"SELECT COALESCE(MAX(fact_job_lifecycle_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
        max_sk = max_sk_result[0]['max_sk']
        
        # Generate new surrogate keys
        window_spec = Window.orderBy("change_id")
        
        facts_to_insert = new_facts.withColumn(
            "fact_job_lifecycle_sk",
            (F.row_number().over(window_spec) + max_sk).cast(LongType())
        ).select(
            "fact_job_lifecycle_sk",
            "change_id",
            "job_sk",
            "company_sk",
            "location_sk",
            "sector_sk",
            "role_sk",
            "source_sk",
            "change_date_sk",
            "change_timestamp",
            "change_type",
            "lifecycle_event_type",
            "days_in_previous_state",
            "batch_id"
        )
        
        # Insert new records (overwrite if old schema, append if new schema)
        write_mode = "overwrite" if not has_new_schema else "append"
        facts_to_insert.write.format("delta").mode(write_mode).option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)
        
        print(f"✓ Inserted {new_facts_count} new fact records")
    
    # Write metadata
    metadata_record = spark.createDataFrame([(
        run_id,
        changes_count,
        new_facts_count,
        skipped_count,
        force_full_refresh,
        datetime.now(),
        "success",
        None
    )], schema=metadata_schema)
    
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    # Return summary
    summary = {
        "status": "success",
        "run_id": run_id,
        "changes_extracted": changes_count,
        "new_facts_inserted": new_facts_count,
        "facts_skipped": skipped_count,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(f"\n{json.dumps(summary, indent=2)}")
    
except Exception as e:
    error_msg = str(e)
    print(f"\n✗ Error: {error_msg}")
    
    # Write error metadata
    metadata_record = spark.createDataFrame([(
        run_id,
        changes_count,
        0,
        skipped_count,
        force_full_refresh,
        datetime.now(),
        "failed",
        error_msg
    )], schema=metadata_schema)
    
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate lifecycle fact table (new schema)
SELECT 
  COUNT(*) as total_lifecycle_events,
  COUNT(DISTINCT job_sk) as unique_jobs,
  lifecycle_event_type,
  COUNT(*) as event_count,
  AVG(days_in_previous_state) as avg_days_in_state,
  MIN(change_timestamp) as earliest_event,
  MAX(change_timestamp) as latest_event
FROM workspace.warehouse.fact_job_lifecycle
GROUP BY lifecycle_event_type
ORDER BY event_count DESC;

In [0]:
%sql
-- Sample recent lifecycle events with dimension context
SELECT 
  f.fact_job_lifecycle_sk,
  j.enterprise_job_id,
  j.title_normalized,
  c.company_name,
  r.role_name,
  s.source_name,
  f.change_timestamp,
  f.lifecycle_event_type,
  f.days_in_previous_state
FROM workspace.warehouse.fact_job_lifecycle f
INNER JOIN workspace.warehouse.dim_job j 
  ON f.job_sk = j.job_sk AND j.is_current = TRUE
INNER JOIN workspace.warehouse.dim_company c 
  ON f.company_sk = c.company_sk
LEFT JOIN workspace.warehouse.dim_role r 
  ON f.role_sk = r.role_sk AND f.role_sk != -1
INNER JOIN workspace.warehouse.dim_source s 
  ON f.source_sk = s.source_sk
ORDER BY f.change_timestamp DESC
LIMIT 20;